The below code is largely copied from this tutorial from https://jonathansoma.com/words/olmocr-on-macos-with-lm-studio.html with slight edits. Before running the code chunks below be sure to start a local server with the olmocr-7b-0225-preview model loaded. This is the only version I have confirmed it with.

In [ ]:
from openai import OpenAI
# initiate server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio", timeout=60)

In [ ]:
from pprint import pprint
from olmocr.pipeline import build_page_query

## construct query 
query = await build_page_query("sample.pdf", page=1,target_longest_image_dim=1024)

query

In [ ]:
query['model'] = 'allenai_olmocr-7b-1025'
response = client.chat.completions.create(**query)

import re

def strip_front_matter(text: str) -> str:
    # remove the first front-matter block delimited by --- lines
    return re.sub(r"^---\s*\n.*?\n---\s*\n", "", text, flags=re.DOTALL)

clean = strip_front_matter(response.choices[0].message.content)
print(clean)

In [18]:
import json
import yaml
from pathlib import Path
from tqdm.notebook import trange

from pypdf import PdfReader
from olmocr.pipeline import build_page_query

async def process_page(filename, page_num):
    """Process a single PDF page and extract its transcription as JSON"""
    query = await build_page_query(filename,
                                   page=page_num,
                                   target_longest_image_dim=1024)
    query['model'] = 'allenai_olmocr-7b-1025'
    response = client.chat.completions.create(**query)
    clean = strip_front_matter(response.choices[0].message.content)
    print(f"Page {page_num} - Content length: {len(clean) if clean else 0}")
    
    # Return JSON object with filename, page_num, text (clean), and token counts
    return {
        "filename": filename,
        "page_num": page_num,
        "text": clean,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

# Process single PDF file
filename = "sample.pdf"
reader = PdfReader(filename)
num_pages = reader.get_num_pages()
results = []

for page_num in trange(1, num_pages + 1):
    result = await process_page(filename, page_num)
    results.append(result)

  0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 1 - Content length: 313


In [20]:
print(results)

[{'filename': 'sample.pdf', 'page_num': 1, 'text': "Name: John Arquette\n\nIndian name: Puyallup\n\nTribe: Puyallup\n\nAge: 19\n\nBlood: \\( \\frac{1}{2} \\)\n\nAgency:\n\nFather: I. Isaac Arquette\n\nArrived: 7-7-'96\n\nDeparted: 7-8-'81\n\nCause: Conduct.\n\nClass entered:\n\nClass left:\n\nTrade:\n\nOuting:\n\nCharacter:\n\nMarried:\n\nDeceased:\n\nRemarks:\nYAWMAN & ERBE MFG. CO., ROCHESTER, N.Y.", 'input_tokens': 945, 'output_tokens': 146}]


In [ ]:
import os
async def process_and_save_directory(input_dir, output_file="double_batch_outputs/output.jsonl"):
    """
    Process all PDF files in a directory and save results to a JSONL file.
    
    Args:
        input_dir: Directory containing PDFs to process
        output_file: Path to output JSONL file (one JSON object per line)
    
    Returns:
        List of result dicts
    """
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith('.pdf')]
    
    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return []
    
    all_results = []
    output_path = Path(output_file)
    output_path.parent.mkdir(exist_ok=True, parents=True)
    
    with open(output_path, 'w') as outf:
        for pdf_file in pdf_files:
            pdf_path = os.path.join(input_dir, pdf_file)
            print(f"\nProcessing: {pdf_file}")
            
            reader = PdfReader(pdf_path)
            num_pages = reader.get_num_pages()
            
            for page_num in trange(1, num_pages + 1):
                result = await process_page(pdf_path, page_num)
                all_results.append(result)
                # Write each result as a line in JSONL file
                outf.write(json.dumps(result) + '\n')
    
    print(f"\nProcessed {len(all_results)} pages from {len(pdf_files)} files")
    print(f"Results saved to {output_path}")
    
    return all_results

In [30]:
await process_and_save_directory("double_batch")


Processing: NARA_1327_b032_f1514.pdf


  0%|          | 0/2 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 1 - Content length: 619


INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 4 - Content length: 1039


INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 5 - Content length: 154

Processed 7 pages from 2 files
Results saved to test_batch_outputs/output.jsonl


[{'filename': 'double_batch/NARA_1327_b032_f1514.pdf',
  'page_num': 1,
  'text': '1514\n\nCARLISLE INDIAN INDUSTRIAL SCHOOL.\nDESCRIPTIVE AND HISTORICAL RECORD OF STUDENT.\n\nNUMBER: 1886\nENGLISH NAME: Joseph Craig\nINDIAN NAME:\nAGENCY: Pogallap\nNATION:\n\nHOME ADDRESS: John Craig\nBLOOD: \nAGE: 16\nHEIGHT: 5\'5"\nWEIGHT: 144\nFORCED INP.: 36%\nFORCED EXPR.: 33\nSEX: M\n\nPARENTS LIVING OR DEAD:\nFATHER: Living\nMOTHER: Living\nARRIVED AT SCHOOL: July 7, 1890\nFOR WHAT PERIOD: 5 years\nDATE DISCHARGED: July 3, 1897\nCAUSE OF DISCHARGE: Conduct\n\nTO COUNTRY:\nFROM COUNTRY:\n\nPATRON\'S NAME AND ADDRESS:\n\nMonths in school before Carlisle,\n\nGrade entered at Carlisle,\n\nGrade at date of Discharge,\n\nTrade or Industry,\n\nChurch.',
  'input_tokens': 945,
  'output_tokens': 253},
 {'filename': 'double_batch/NARA_1327_b032_f1514.pdf',
  'page_num': 2,
  'text': '1574. REPORT AFTER LEAVING CARLISLE\n\nNAME AT CARLISLE: Joseph Craig\n\nPRESENT NAME:\n\nDATE: 1910\nINFORMATION THROUGH

In [42]:
import ipywidgets as widgets
from IPython.display import HTML, Image as IPImage
from pathlib import Path
import json
import csv
from io import BytesIO

class JSONLViewer:
    def __init__(self, jsonl_file, quality_csv="workspace/quality_ratings.csv"):
        """
        Initialize viewer for JSONL transcription results.
        
        Args:
            jsonl_file: Path to JSONL file with results
            quality_csv: Path to CSV file for storing quality ratings
        """
        self.jsonl_file = jsonl_file
        self.quality_csv = quality_csv
        self.current_index = 0
        self.ratings = {}
        
        # Load JSONL data
        self.results = []
        with open(jsonl_file, 'r') as f:
            for line in f:
                if line.strip():
                    self.results.append(json.loads(line))
        
        # Load existing ratings if CSV exists
        if Path(quality_csv).exists():
            with open(quality_csv, 'r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    key = f"{row['filename']}_{row['page_num']}"
                    self.ratings[key] = row['quality']
        
        self.build_ui()
    
    def build_ui(self):
        """Build the interactive UI."""
        # Navigation
        self.nav_label = widgets.HTML(f"<h3>Page 1 of {len(self.results)}</h3>")
        prev_btn = widgets.Button(description='← Previous')
        next_btn = widgets.Button(description='Next →')
        prev_btn.on_click(self.prev_page)
        next_btn.on_click(self.next_page)
        nav_box = widgets.HBox([prev_btn, self.nav_label, next_btn])
        
        # Display widgets for PDF and text
        self.pdf_widget = widgets.Image()
        self.text_widget = widgets.HTML()
        
        # Quality rating buttons
        good_btn = widgets.Button(description='✓ Good', button_style='success')
        bad_btn = widgets.Button(description='✗ Bad', button_style='danger')
        skip_btn = widgets.Button(description='Skip')
        self.rating_label = widgets.HTML("")
        
        good_btn.on_click(lambda b: self.rate_quality('good'))
        bad_btn.on_click(lambda b: self.rate_quality('bad'))
        skip_btn.on_click(lambda b: self.next_page(None))
        
        rating_box = widgets.HBox([good_btn, bad_btn, skip_btn])
        
        # Assemble main layout
        self.main_box = widgets.VBox([
            nav_box,
            self.pdf_widget,
            self.text_widget,
            self.rating_label,
            rating_box
        ])
        
        self.refresh_display()
    
    def refresh_display(self):
        """Update the display for the current result."""
        if self.current_index >= len(self.results):
            self.current_index = len(self.results) - 1
        
        result = self.results[self.current_index]
        filename = result['filename']
        page_num = result['page_num']
        text = result.get('text', '')
        
        # Update navigation label
        self.nav_label.value = f"<h3>Page {self.current_index + 1} of {len(self.results)}</h3>"
        
        # Update PDF image widget
        try:
            from pdf2image import convert_from_path
            images = convert_from_path(filename, first_page=page_num, last_page=page_num)
            if images:
                # Convert PIL image to bytes
                img_bytes = BytesIO()
                images[0].save(img_bytes, format='PNG')
                img_bytes.seek(0)
                self.pdf_widget.value = img_bytes.read()
            else:
                self.pdf_widget.value = b''
        except Exception as e:
            self.pdf_widget.value = b''
            print(f"Error loading PDF: {str(e)}\nFilepath: {filename}")
        
        # Update text widget
        self.text_widget.value = f"""
        <div style="border: 1px solid #ccc; padding: 10px; max-height: 400px; overflow-y: auto; background: #f9f9f9; margin-top: 20px;">
            <h4>Extracted Text (Page {page_num})</h4>
            <pre style="white-space: pre-wrap; font-family: monospace; font-size: 11px; margin: 0;">{text}</pre>
        </div>
        """
        
        # Update rating label
        key = f"{filename}_{page_num}"
        rating = self.ratings.get(key, "Not rated")
        self.rating_label.value = f"<p><strong>Rating:</strong> {rating}</p>"
    
    def rate_quality(self, quality):
        """Save quality rating and move to next."""
        result = self.results[self.current_index]
        filename = result['filename']
        page_num = result['page_num']
        
        key = f"{filename}_{page_num}"
        self.ratings[key] = quality
        self.save_ratings()
        
        print(f"✓ Rated '{quality}' for {filename} page {page_num}")
        self.next_page(None)
    
    def save_ratings(self):
        """Save all ratings to CSV."""
        output_path = Path(self.quality_csv)
        output_path.parent.mkdir(exist_ok=True, parents=True)
        
        with open(output_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['filename', 'page_num', 'quality'])
            writer.writeheader()
            
            for key, quality in self.ratings.items():
                parts = key.rsplit('_', 1)
                if len(parts) == 2:
                    fn, pn = parts
                    writer.writerow({'filename': fn, 'page_num': pn, 'quality': quality})
    
    def prev_page(self, b):
        """Go to previous page."""
        if self.current_index > 0:
            self.current_index -= 1
            self.refresh_display()
    
    def next_page(self, b):
        """Go to next page."""
        if self.current_index < len(self.results) - 1:
            self.current_index += 1
            self.refresh_display()
    
    def show(self):
        """Display the viewer."""
        from IPython.display import display
        display(self.main_box)

In [43]:
viewer = JSONLViewer("test_batch_outputs/output.jsonl")
viewer.show()